### Imports & chemins

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments
)
from trl import SFTTrainer
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

# Résoudre les chemins automatiquement
NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))
DATASET_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "dataset.jsonl")
MODEL_OUTPUT = os.path.join(PROJECT_ROOT, "models", "lora_checkpoint")

print("Notebook dir :", NOTEBOOK_DIR)
print("Project root  :", PROJECT_ROOT)
print("Dataset path  :", DATASET_PATH)
print("Output path   :", MODEL_OUTPUT)


### Charger le dataset

In [ ]:
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

print(dataset)
print("\nExemple :")
print(dataset[0])


### Choisir le modèle de base

In [ ]:
#  Choix ton modèle  :
BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"
# BASE_MODEL = "meta-llama/Meta-Llama-3-8B"

print("Using model:", BASE_MODEL)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token


### Charger le modèle + préparer LoRA

In [ ]:
bnb_config = dict(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    **bnb_config
)

model = prepare_model_for_kbit_training(model)

# Configuration LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"]  # modules ciblés
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()


### Préparer le format d'entraînement

In [ ]:
def format_training(example):
    return {
        "text": f"<s>[USER] {example['input']}\n[ASSISTANT] {example['response']}</s>"
    }

train_dataset = dataset.map(format_training)

print(train_dataset[0])


### Configuration de l’entraînement

In [ ]:
training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    max_steps=200,          #  À augmenter pour un vrai modèle
    logging_steps=10,
    learning_rate=2e-4,
    fp16=True,
    save_strategy="steps",
    save_steps=50,
    optim="paged_adamw_8bit"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
    dataset_text_field="text",
    args=training_args,
)


### Lancer l’entraînement LoRA

In [ ]:
trainer.train()

model.save_pretrained(MODEL_OUTPUT)
tokenizer.save_pretrained(MODEL_OUTPUT)

print("✔ Modèle LoRA sauvegardé dans :", MODEL_OUTPUT)


### Tester rapidement le modèle

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=MODEL_OUTPUT,
    tokenizer=tokenizer,
    device_map="auto",
    max_new_tokens=200
)

prompt = "Quels sont les documents nécessaires pour un contrat habitation ?"

res = pipe(prompt)[0]["generated_text"]
print(res)
